<a href="https://colab.research.google.com/github/pushpakgaidhane294/Gen_AI_Lab/blob/main/Practical4_Gen_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


# =====================================================
# 1. Small English-Hindi dataset
# =====================================================

english = [
    "hello",
    "how are you",
    "i am fine",
    "thank you",
    "good morning",
    "good night",
    "what is your name",
    "my name is ram",
    "i am happy",
    "i love india"
]

hindi = [
    "नमस्ते",
    "आप कैसे हैं",
    "मैं ठीक हूँ",
    "धन्यवाद",
    "सुप्रभात",
    "शुभ रात्रि",
    "आपका नाम क्या है",
    "मेरा नाम राम है",
    "मैं खुश हूँ",
    "मुझे भारत पसंद है"
]


# Add start and end tokens
hindi = [
    "start_ " + sentence + " _end"
    for sentence in hindi
]


# =====================================================
# 2. Tokenization
# =====================================================

eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english)

hin_tokenizer = Tokenizer(
    filters=""
)
hin_tokenizer.fit_on_texts(hindi)


eng_sequences = eng_tokenizer.texts_to_sequences(
    english
)

hin_sequences = hin_tokenizer.texts_to_sequences(
    hindi
)


eng_vocab_size = len(
    eng_tokenizer.word_index
) + 1

hin_vocab_size = len(
    hin_tokenizer.word_index
) + 1


# =====================================================
# 3. Padding
# =====================================================

max_eng_len = max(
    len(x) for x in eng_sequences
)

max_hin_len = max(
    len(x) for x in hin_sequences
)


encoder_input = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding="post"
)

decoder_input = pad_sequences(
    hin_sequences,
    maxlen=max_hin_len,
    padding="post"
)


# Decoder target is shifted by one word
decoder_target = np.zeros_like(
    decoder_input
)

decoder_target[:, :-1] = decoder_input[:, 1:]


# =====================================================
# 4. Encoder
# =====================================================

latent_dim = 128

encoder_inputs = Input(
    shape=(None,)
)

encoder_embedding = Embedding(
    eng_vocab_size,
    latent_dim
)

encoder_embedded = encoder_embedding(
    encoder_inputs
)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

_, state_h, state_c = encoder_lstm(
    encoder_embedded
)

encoder_states = [
    state_h,
    state_c
]


# =====================================================
# 5. Decoder
# =====================================================

decoder_inputs = Input(
    shape=(None,)
)

decoder_embedding = Embedding(
    hin_vocab_size,
    latent_dim
)

decoder_embedded = decoder_embedding(
    decoder_inputs
)

decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedded,
    initial_state=encoder_states
)

decoder_dense = Dense(
    hin_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(
    decoder_outputs
)


# =====================================================
# 6. Create model
# =====================================================

model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


# =====================================================
# 7. Train
# =====================================================

model.fit(
    [encoder_input, decoder_input],
    decoder_target[..., np.newaxis],
    batch_size=2,
    epochs=300,
    verbose=0
)

print("Training completed!")


# =====================================================
# 8. Encoder for prediction
# =====================================================

encoder_model = Model(
    encoder_inputs,
    encoder_states
)


# =====================================================
# 9. Decoder for prediction
# =====================================================

decoder_state_h = Input(
    shape=(latent_dim,)
)

decoder_state_c = Input(
    shape=(latent_dim,)
)

decoder_states_inputs = [
    decoder_state_h,
    decoder_state_c
]

decoder_embedded_inf = decoder_embedding(
    decoder_inputs
)

decoder_outputs_inf, h, c = decoder_lstm(
    decoder_embedded_inf,
    initial_state=decoder_states_inputs
)

decoder_outputs_inf = decoder_dense(
    decoder_outputs_inf
)

decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs_inf, h, c]
)


# =====================================================
# 10. Reverse Hindi dictionary
# =====================================================

reverse_hindi = {
    index: word
    for word, index
    in hin_tokenizer.word_index.items()
}

start_token = hin_tokenizer.word_index["start_"]
end_token = hin_tokenizer.word_index["_end"]


# =====================================================
# 11. Translation function
# =====================================================

def translate(sentence):

    # Convert English sentence to numbers
    sequence = eng_tokenizer.texts_to_sequences(
        [sentence.lower()]
    )

    sequence = pad_sequences(
        sequence,
        maxlen=max_eng_len,
        padding="post"
    )

    # Encode
    states = encoder_model.predict(
        sequence,
        verbose=0
    )

    # Start decoder with START token
    target = np.array(
        [[start_token]]
    )

    result = []

    for _ in range(max_hin_len):

        output, h, c = decoder_model.predict(
            [target] + states,
            verbose=0
        )

        token = np.argmax(
            output[0, -1, :]
        )

        if token == end_token:
            break

        word = reverse_hindi.get(
            token,
            ""
        )

        if word != "start_":
            result.append(word)

        target = np.array(
            [[token]]
        )

        states = [h, c]

    return " ".join(result)


# =====================================================
# 12. Test
# =====================================================

print("\nTranslation Results")
print("-------------------")

test_sentences = [
    "hello",
    "My name is Ram",
    "good morning",
    "how are you",
    "i am  fine"
]

for sentence in test_sentences:

    print(
        sentence,
        "->",
        translate(sentence)
    )

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_5       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 128) │      2,688 │ input_layer_4[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 128) │      3,072 │ input_layer_5[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, 128),     │    131,584 │ embedding_2[0][0] │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │    131,584 │ embedding_3[0][0… │
│                     │ 128), (None,      │            │ lstm_2[0][1],     │
│                     │ 128), (None,      │            │ lstm_2[0][2]      │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 24)  │      3,096 │ lstm_3[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 272,024 (1.04 MB)

 Trainable params: 272,024 (1.04 MB)

 Non-trainable params: 0 (0.00 B)